# TP3 — Clustering et word2vec
Corpus : journal *Le Soir* (CAMille), décennie **1940**.

**Partie 1** : clustering de pages (TF-IDF + KMeans). **Partie 2** : word2vec sur `sents.txt`.

> Les étapes longues (écriture du corpus avec expressions, entraînement) sont ignorées si leurs fichiers existent déjà dans `data/`.

## Partie 1 — Clustering des pages de 1940

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from nltk.tokenize import wordpunct_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

### 1. Chargement des pages de la décennie

In [ ]:
DECADE = '1940'
data_path = "../../data/txt/"
files = sorted(f for f in os.listdir(data_path)
               if f.endswith(".txt") and f.startswith(f"KB_JB838_{DECADE[:3]}"))
texts = []
for f in files:
    with open(os.path.join(data_path, f), encoding="utf-8") as fh:
        texts.append(fh.read())

meta = pd.DataFrame({"file": files})
meta["year"] = meta["file"].apply(lambda f: int(f.split("_")[2][:4]))
meta["page"] = meta["file"].apply(lambda f: int(f.split("-")[-1].split(".")[0]))
print(len(files), "pages")
print(meta["year"].value_counts().sort_index().to_dict())

### 2. Vectorisation TF-IDF
Tokenisation avec `wordpunct_tokenize` (sans `punkt`), mots alphabétiques seulement, stopwords français, `max_df=0.5` et `min_df=0.1`.

In [ ]:
def preprocessing(text):
    return [t for t in wordpunct_tokenize(text) if t.isalpha()]

vectorizer = TfidfVectorizer(tokenizer=preprocessing, token_pattern=None,
                             stop_words=stopwords.words('french'),
                             max_df=0.5, min_df=0.1, lowercase=True)
tfidf_vectors = vectorizer.fit_transform(texts)
terms = vectorizer.get_feature_names_out()
print(tfidf_vectors.shape)

### 3. Choix du nombre de clusters (inertie et silhouette, k = 2 à 10)

In [ ]:
res = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(tfidf_vectors)
    res.append((k, km.inertia_, silhouette_score(tfidf_vectors, km.labels_)))
res_df = pd.DataFrame(res, columns=["k", "inertie", "silhouette"]).round(3)
print(res_df.to_string(index=False))

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(res_df["k"], res_df["inertie"], marker="o"); ax[0].set_title("Inertie"); ax[0].set_xlabel("k")
ax[1].plot(res_df["k"], res_df["silhouette"], marker="o"); ax[1].set_title("Silhouette"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.savefig("choix_k_1940.png", dpi=200, bbox_inches="tight"); plt.show()

### 4. Clustering final : k = 5
`n_init=10` et `random_state=42` rendent les étiquettes reproductibles.

In [ ]:
N_CLUSTERS = 5
km_model = KMeans(n_clusters=N_CLUSTERS, n_init=10, random_state=42)
clusters = km_model.fit_predict(tfidf_vectors)
meta["cluster"] = clusters
print("Tailles :", np.bincount(clusters))

### 5. Interprétation : termes les plus importants par cluster

In [ ]:
order_centroids = km_model.cluster_centers_.argsort()[:, ::-1]
for i in range(N_CLUSTERS):
    print(f"Cluster {i} ({(clusters == i).sum()} pages) :", ", ".join(terms[j] for j in order_centroids[i, :12]))

### 6. Répartition par année et par page

In [ ]:
print(pd.crosstab(meta["year"], meta["cluster"]))
print()
print(meta.groupby("cluster")["page"].describe()[["count", "mean", "min", "50%", "max"]].round(1))

### 7. Extraits (milieu de page) de 4 pages tirées au hasard dans chaque cluster

In [ ]:
random.seed(42)
for i in range(N_CLUSTERS):
    idx = [j for j in range(len(files)) if clusters[j] == i]
    print(f"\n===== Cluster {i} ({len(idx)} pages) =====")
    for j in random.sample(idx, min(4, len(idx))):
        mid = len(texts[j]) // 2
        print(f"- {files[j]} (page {meta.loc[j, 'page']}) : {texts[j][mid:mid + 200]!r}")

### 8. Projection PCA en 2D

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(tfidf_vectors.toarray())
centers = pca.transform(km_model.cluster_centers_)
print("Variance expliquée par les 2 axes :", pca.explained_variance_ratio_.round(3), "->", round(pca.explained_variance_ratio_.sum(), 3))

fig, ax = plt.subplots(figsize=(10, 10))
sc = ax.scatter(coords[:, 0], coords[:, 1], c=clusters, s=90)
ax.scatter(centers[:, 0], centers[:, 1], c="black", marker="x", s=100, linewidths=2)
ax.legend(*sc.legend_elements(), title="Clusters")
plt.savefig("clusters_1940.png", dpi=200, bbox_inches="tight")
plt.show()

### 9. Nuages de mots par cluster

In [ ]:
try:
    from wordcloud import WordCloud
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for i, ax in enumerate(axes.ravel()):
        if i < N_CLUSTERS:
            weights = {terms[j]: km_model.cluster_centers_[i, j] for j in order_centroids[i, :60]}
            ax.imshow(WordCloud(width=600, height=400, background_color="white").generate_from_frequencies(weights))
            ax.set_title(f"Cluster {i}")
        ax.axis("off")
    plt.tight_layout(); plt.savefig("nuages_clusters_1940.png", dpi=200, bbox_inches="tight"); plt.show()
except ImportError:
    print("wordcloud non disponible")

### 10. Interprétation des clusters

| Cluster | Taille | Pages | Contenu observé |
|---|---|---|---|
| 0 | 54 | 13-14 | annonces automobiles et d'occasion (cluster net) |
| 1 | 220 | 4-10 | petites annonces générales : « Agence Rossel », villas, terrains, emplois (net) |
| 2 | 433 | 1-4 | contenu rédactionnel général : politique, chroniques, sport local, météo (hétérogène) |
| 3 | 39 | 2-7 | annonces notariales (ventes, terrains à bâtir), avec une exception (fait divers) |
| 4 | 254 | 3-11 | cours de Bourse, résultats sportifs, feuilleton (mixte, tableaux chiffrés) |

Les rubriques au vocabulaire codé (annonces) sont bien isolées par TF-IDF. Le contenu rédactionnel est plus hétérogène : les pages sont entières et mêlent plusieurs rubriques, l'OCR est bruité et la silhouette reste faible (~0,1).

## Partie 2 — word2vec sur `sents.txt`

In [ ]:
import time, unicodedata
from gensim.models import Word2Vec
from gensim.models.phrases import Phrases, Phraser
from gensim.models.word2vec import LineSentence

infile = "../../data/sents.txt"

def norm(w):
    w = unicodedata.normalize("NFKD", w.lower())
    return "".join(c for c in w if not unicodedata.combining(c))

class MySentences:
    def __init__(self, path, step=1):
        self.path, self.step = path, step
    def __iter__(self):
        with open(self.path, encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i % self.step:
                    continue
                toks = [norm(t) for t in wordpunct_tokenize(line) if t.isalpha()]
                if len(toks) >= 3:
                    yield toks

### 1. Échantillon (1 phrase sur 10) pour comparer les réglages
Le fichier est trié chronologiquement : l'échantillon couvre toutes les époques.

In [ ]:
t0 = time.time()
sample = list(MySentences(infile, step=10))
print(len(sample), "phrases,", sum(len(s) for s in sample), "tokens,", f"{time.time() - t0:.0f} s")

### 2. Expressions (bigrammes et trigrammes) et comparaison de plusieurs modèles

In [ ]:
t0 = time.time()
bigram = Phraser(Phrases(sample, min_count=20))
trigram = Phraser(Phrases(bigram[sample], min_count=20))
corpus_sample = [trigram[bigram[s]] for s in sample]
print(f"Phrases : {time.time() - t0:.0f} s")

configs = [{"window": 3, "min_count": 20}, {"window": 5, "min_count": 20},
           {"window": 10, "min_count": 20}, {"window": 5, "min_count": 50}]
models = {}
for cfg in configs:
    t0 = time.time()
    m = Word2Vec(corpus_sample, vector_size=100, window=cfg["window"], min_count=cfg["min_count"],
                 workers=os.cpu_count(), epochs=5, seed=42)
    name = f"w{cfg['window']}_mc{cfg['min_count']}"
    models[name] = m
    print(f"\n=== {name} : {len(m.wv)} mots, {time.time() - t0:.0f} s ===")
    for mot in ["ministre", "guerre"]:
        print(mot, "->", [w for w, _ in m.wv.most_similar(mot, topn=6)])

### 3. Corpus complet avec expressions
Les phraseurs appris sur l'échantillon sont appliqués à tout `sents.txt`. Le résultat est écrit une fois sur disque (environ 5 minutes).

In [ ]:
out = "../../data/sents_phrased.txt"
if not os.path.exists(out):
    t0 = time.time()
    n = 0
    with open(out, "w", encoding="utf-8") as f:
        for s in MySentences(infile, step=1):
            f.write(" ".join(trigram[bigram[s]]) + "\n")
            n += 1
            if n % 1_000_000 == 0:
                print(n, f"{time.time() - t0:.0f} s")
    print(n, "phrases écrites en", f"{time.time() - t0:.0f} s")
else:
    print("Corpus déjà écrit :", out)

### 4. Modèles finaux (fenêtres 5 et 10, min_count = 20)

In [ ]:
corpus_full = LineSentence(out)
final = {}
for window in (5, 10):
    path = f"../../data/newspapers_w{window}.model"
    t0 = time.time()
    if os.path.exists(path):
        final[window] = Word2Vec.load(path)
    else:
        final[window] = Word2Vec(corpus_full, vector_size=100, window=window, min_count=20,
                                 workers=os.cpu_count(), epochs=5, seed=42)
        final[window].save(path)
    print(f"window={window} : {len(final[window].wv)} mots, {time.time() - t0:.0f} s")

### 5. Exploration : `most_similar`, `similarity` et analogie

In [ ]:
def voisins(mot, topn=8):
    for w in (5, 10):
        wv = final[w].wv
        if mot in wv:
            print(f"  window={w:<2} :", [(m, round(s, 2)) for m, s in wv.most_similar(mot, topn=topn)])
        else:
            print(f"  window={w:<2} : '{mot}' absent du vocabulaire")

def sim(a, b):
    res = []
    for w in (5, 10):
        wv = final[w].wv
        res.append(round(float(wv.similarity(a, b)), 3) if a in wv and b in wv else None)
    print(f"{a} / {b} : window5={res[0]}, window10={res[1]}")

print("===== most_similar =====")
for mot in ["ministre", "guerre", "roi", "paris", "bruxelles", "chomage", "tramway"]:
    print(mot)
    voisins(mot)

print("\n===== similarity =====")
for a, b in [("ministre", "roi"), ("paris", "bruxelles"), ("guerre", "paix"),
             ("homme", "femme"), ("guerre", "chocolat")]:
    sim(a, b)

print("\n===== analogie : paris - france + belgique =====")
for w in (5, 10):
    wv = final[w].wv
    try:
        print(f"  window={w:<2} :", wv.most_similar(positive=["paris", "belgique"], negative=["france"], topn=5))
    except KeyError as e:
        print(f"  window={w:<2} : mot absent ({e})")

### 6. Discussion

**Réglages.** Sur l'échantillon (≈9 M de tokens, 29 224 mots), une fenêtre de 3 donne des voisins de type fonctionnel (`secretaire_general`, `vice_president`), alors qu'une fenêtre de 10 donne des voisins plus thématiques (`l_armistice`, `les_allies` pour « guerre »). `min_count=50` divise le vocabulaire par deux (14 646 mots) sans améliorer les voisins. Sur le corpus complet, les modèles ont 109 833 mots.

**Résultats.** Les voisins sont cohérents : `ministre` → `premier_ministre`, `president_du_conseil` ; `roi` → `prince`, `pape`, `tsar`, `sultan`, `roi_albert` ; `paris` → grandes villes ; `chomage` → `chomeurs`, `licenciement`, `allocations` ; `tramway` → `tram_vicinal`, `autobus`. `guerre / paix` ≈ 0,5 (mêmes contextes) et `guerre / chocolat` ≈ −0,06.

**Limites.** L'OCR laisse des variantes comme `guerro`, `guorre` ou `bru_xelles`, que le modèle rapproche du mot correct. Cette dispersion explique probablement la faible similarité `paris / bruxelles` (0,37–0,49) et l'échec de l'analogie `paris − france + belgique` (le résultat attendu, `bruxelles`, n'apparaît pas). `Phrases` produit aussi des expressions parasites (`le_ministre`) et recolle les élisions (`l_occupation`).